In [ ]:
import pickle, os
from lowresource_llm_evaluation.LanguageDatasets import LanguageDataset
from transformers import AutoTokenizer
import numpy as np

MODELO = "Qwen/Qwen2.5-7B-Instruct" 
TOKENIZER = AutoTokenizer.from_pretrained(MODELO, trust_remote_code=True)

def compute_lengths(dataset, tokenizer):
    def _len(batch):
        tok = tokenizer(batch["text"], truncation=False)
        return {"len": [len(x) for x in tok["input_ids"]]}
    return dataset.map(_len, batched=True)["len"]


def saveTrainTest(language, modelo, tokenizer, thr=0.3, max_length=None, compute_length=False, N_max=20000):
    root_dir = "TrainDatasets"
    os.makedirs(root_dir, exist_ok=True)

    save_dir = f"{root_dir}/{modelo}"
    os.makedirs(save_dir, exist_ok=True)

    # Dataset sin tokenizar
    ds = (
        LanguageDataset(language, filter_language_thr=thr)
        .read_opus(source="NLLB", version=1)
        .filter_by_language(1024, top_k=10)
    )

    # Calcular longitudes ANTES de tokenizar
    if compute_length:
        subset = ds.hf_dataset.select(range(min(N_max, len(ds.hf_dataset))))
        lengths = compute_lengths(subset, tokenizer)
        print("Longitud media de las líneas:", sum(lengths)/len(lengths))
        for n in [75, 90, 95, 99]:
            print(f"Percentil {n} de longitud:",np.percentile(lengths, n)) 
        print("Longitud máxima: ",max(lengths))  
    if not max_length:
        max_length = int(np.percentile(lengths, 95)) 
    # Tokenizar y dividir
    train, test = ds.split(test_size=0.05, tokenizer=tokenizer, max_length=max_length)

    # Guardar
    save_path = f"{save_dir}/{language}.pkl"
    with open(save_path, "wb") as f:
        pickle.dump((train, test), f, protocol=5)

    print(f"Dataset guardado correctamente en: {save_path}")
    return train, test

In [4]:
language = "asturiano"
train, test = saveTrainTest(language, MODELO, TOKENIZER, 0.4, compute_length=True)

Empezando descarga
Descarga completada. 
Procesando las líneas
Dataset cargado


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Longitud media de las líneas: 42.97705
Percentil {n} de longitud: 54.0
Percentil {n} de longitud: 77.0
Percentil {n} de longitud: 91.0
Percentil {n} de longitud: 125.0
Longitud máxima:  167


Map:   0%|          | 0/682820 [00:00<?, ? examples/s]

Map:   0%|          | 0/35938 [00:00<?, ? examples/s]

Dataset guardado correctamente en: TrainDatasets/Qwen/Qwen2.5-7B-Instruct/asturiano.pkl


In [5]:
language = "aranes"
saveTrainTest(language, MODELO, TOKENIZER, 0.4)

Empezando descarga
Descarga completada. 
Procesando las líneas


KeyboardInterrupt: 